In [77]:
import pandas as pd
import numpy as np

In [78]:
df = pd.read_csv(r"C:\Users\TGX-100\Documents\GitHub\Urban-Traffic-Flow-Prediction\dataset\processed\GA0151_intersection.csv")

In [79]:
df['date'] = pd.to_datetime(df['date'])

In [80]:
df.head()

,date,time,GA0151_A,GA0151_C,GA0151_D
0,2019-10-01,0,6,16,15
1,2019-10-01,1,4,16,8
2,2019-10-01,2,4,12,14
3,2019-10-01,3,0,16,10
4,2019-10-01,4,4,21,15


## Time based features

In [81]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [84]:
df.rename(columns={'time':'hour'}, inplace=True)

In [85]:
df

,date,hour,GA0151_A,GA0151_C,GA0151_D,day_of_week,month,year,is_weekend
0,2019-10-01,0,6,16,15,1,10,2019,0
1,2019-10-01,1,4,16,8,1,10,2019,0
2,2019-10-01,2,4,12,14,1,10,2019,0
3,2019-10-01,3,0,16,10,1,10,2019,0
4,2019-10-01,4,4,21,15,1,10,2019,0
...,...,...,...,...,...,...,...,...,...
33639,2023-09-30,19,37,205,146,5,9,2023,1
33640,2023-09-30,20,33,159,154,5,9,2023,1
33641,2023-09-30,21,43,152,145,5,9,2023,1
33642,2023-09-30,22,36,159,119,5,9,2023,1


## Cyclical Time features

In [86]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [87]:
df

,date,hour,GA0151_A,GA0151_C,GA0151_D,day_of_week,month,year,is_weekend,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos
0,2019-10-01,0,6,16,15,1,10,2019,0,0.000000,1.000000,0.781831,0.623490,-0.866025,5.000000e-01
1,2019-10-01,1,4,16,8,1,10,2019,0,0.258819,0.965926,0.781831,0.623490,-0.866025,5.000000e-01
2,2019-10-01,2,4,12,14,1,10,2019,0,0.500000,0.866025,0.781831,0.623490,-0.866025,5.000000e-01
3,2019-10-01,3,0,16,10,1,10,2019,0,0.707107,0.707107,0.781831,0.623490,-0.866025,5.000000e-01
4,2019-10-01,4,4,21,15,1,10,2019,0,0.866025,0.500000,0.781831,0.623490,-0.866025,5.000000e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33639,2023-09-30,19,37,205,146,5,9,2023,1,-0.965926,0.258819,-0.974928,-0.222521,-1.000000,-1.836970e-16
33640,2023-09-30,20,33,159,154,5,9,2023,1,-0.866025,0.500000,-0.974928,-0.222521,-1.000000,-1.836970e-16
33641,2023-09-30,21,43,152,145,5,9,2023,1,-0.707107,0.707107,-0.974928,-0.222521,-1.000000,-1.836970e-16
33642,2023-09-30,22,36,159,119,5,9,2023,1,-0.500000,0.866025,-0.974928,-0.222521,-1.000000,-1.836970e-16


### Lag Features for sensor

In [88]:
sensors = ["GA0151_A", "GA0151_C", "GA0151_D"]

for sensor in sensors:
    # Lag Features
    df[f"{sensor}_lag_1"] = df[sensor].shift(1)
    df[f"{sensor}_lag_2"] = df[sensor].shift(2)
    df[f"{sensor}_lag_3"] = df[sensor].shift(3)
    df[f"{sensor}_lag_6"] = df[sensor].shift(6)
    df[f"{sensor}_lag_12"] = df[sensor].shift(12)
    df[f"{sensor}_lag_24"] = df[sensor].shift(24)
    df[f"{sensor}_lag_48"] = df[sensor].shift(48)
    df[f"{sensor}_lag_168"] = df[sensor].shift(168)

    # Rolling Mean
    df[f"{sensor}_rolling_mean_3"] = df[sensor].shift(1).rolling(3).mean()
    df[f"{sensor}_rolling_mean_6"] = df[sensor].shift(1).rolling(6).mean()
    df[f"{sensor}_rolling_mean_12"] = df[sensor].shift(1).rolling(12).mean()
    df[f"{sensor}_rolling_mean_24"] = df[sensor].shift(1).rolling(24).mean()

    # Rolling Standard Deviation
    df[f"{sensor}_rolling_std_6"] = df[sensor].shift(1).rolling(6).std()
    df[f"{sensor}_rolling_std_24"] = df[sensor].shift(1).rolling(24).std()

    # Difference Features
    df[f"{sensor}_diff_1"] = df[sensor] - df[f"{sensor}_lag_1"]
    df[f"{sensor}_diff_24"] = df[sensor] - df[f"{sensor}_lag_24"]



### Neighbour Sensor Features

In [89]:
for sensor in sensors:
    other_sensors = [s for s in sensors if s != sensor]
    df[f"{sensor}_neighbor_mean"] = df[other_sensors].mean(axis=1)

### Remove unnecessary columns

In [90]:
df = df.dropna().reset_index(drop=True)


In [91]:
df = df.drop(columns='date')

In [92]:
df.head()

,hour,GA0151_A,GA0151_C,GA0151_D,day_of_week,month,year,is_weekend,hour_sin,hour_cos,...,GA0151_D_rolling_mean_6,GA0151_D_rolling_mean_12,GA0151_D_rolling_mean_24,GA0151_D_rolling_std_6,GA0151_D_rolling_std_24,GA0151_D_diff_1,GA0151_D_diff_24,GA0151_A_neighbor_mean,GA0151_C_neighbor_mean,GA0151_D_neighbor_mean
0,0,12,22,11,1,10,2019,0,0.000000,1.000000,...,70.666667,108.416667,64.958333,40.346830,60.796510,-9.0,-6.0,16.5,11.5,17.0
1,1,4,16,10,1,10,2019,0,0.258819,0.965926,...,53.166667,100.083333,64.708333,39.514132,61.014239,-1.0,-6.0,13.0,7.0,10.0
2,2,5,24,12,1,10,2019,0,0.500000,0.866025,...,36.666667,90.000000,64.458333,31.366649,61.234389,2.0,3.0,18.0,8.5,14.5
3,3,3,12,6,1,10,2019,0,0.707107,0.707107,...,24.500000,78.666667,64.583333,21.463923,61.119211,-6.0,-1.0,9.0,4.5,7.5
4,4,6,38,10,1,10,2019,0,0.866025,0.500000,...,14.500000,65.333333,64.541667,8.043631,61.160501,4.0,0.0,24.0,8.0,22.0
